# 04 - Procesamiento Big Data con PySpark

En este notebook se utilizará Apache Spark mediante PySpark para procesar el dataset limpio del proyecto EdTech Learning Analytics.

Se realizarán las actividades solicitadas:

- carga del dataset limpio en Spark,
- exploración básica,
- transformaciones con PySpark,
- consulta mediante Spark SQL,
- comparación de rendimiento entre Pandas y Spark,
- almacenamiento en formato Parquet,
- documentación de ventajas y limitaciones,
- y representación de la arquitectura del proyecto.

## 1. Importación de librerías

Se cargan las librerías necesarias para trabajar con Spark, Pandas, medición de tiempos y manejo de rutas.

In [1]:
import pandas as pd
import time

from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    avg,
    count,
    desc
)

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [2]:
import sys
import pyspark

print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
print("Ejecutable:", sys.executable)

Python: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
PySpark: 4.3.0
Ejecutable: c:\venvs\edtech-spark\Scripts\python.exe


## 2. Creación de la sesión de Spark

Se crea una sesión local de Apache Spark utilizando los recursos disponibles del equipo.

In [3]:
spark = (
    SparkSession.builder
    .appName("EdTechLearningAnalytics_BigData")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print("Spark session creada correctamente")
print("Versión de Spark:", spark.version)

c:\venvs\edtech-spark\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark session creada correctamente
Versión de Spark: 4.3.0


## 3. Configuración de rutas

Se definen las rutas del proyecto para cargar el dataset limpio generado en la etapa anterior y almacenar posteriormente los resultados de Spark.

In [4]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "datos"
DATA_PROCESSED = DATA_DIR / "procesados"

ruta_dataset_limpio = (
    DATA_PROCESSED / "edtech_dataset_limpio.csv"
)

ruta_parquet = (
    DATA_PROCESSED / "edtech_dataset_spark.parquet"
)

print("Dataset limpio:", ruta_dataset_limpio)
print("Salida Parquet:", ruta_parquet)

Dataset limpio: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\procesados\edtech_dataset_limpio.csv
Salida Parquet: c:\Users\Rocket Consulting\Desktop\edtech_learning_analytics\datos\procesados\edtech_dataset_spark.parquet


In [5]:
print(
    "Dataset encontrado correctamente"
    if ruta_dataset_limpio.exists()
    else "Dataset NO encontrado"
)

Dataset encontrado correctamente


## 4. Carga del dataset limpio en Spark

Se carga `edtech_dataset_limpio.csv` como un DataFrame de Spark utilizando encabezados e inferencia automática del esquema.

In [6]:
df_spark = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(ruta_dataset_limpio))
)

print("Dataset cargado correctamente en Spark")

Dataset cargado correctamente en Spark


## 5. Exploración básica del DataFrame

Se revisa la estructura general del DataFrame de Spark mediante su esquema, una muestra de registros y estadísticas descriptivas.

In [7]:
df_spark.printSchema()

root
 |-- interaction_id: string (nullable = true)
 |-- student_id: string (nullable = true)
 |-- course_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- activity_type: string (nullable = true)
 |-- duration_seconds: integer (nullable = true)
 |-- completion_rate: double (nullable = true)
 |-- score: double (nullable = true)
 |-- attempts: integer (nullable = true)
 |-- device_used: string (nullable = true)
 |-- help_sought: boolean (nullable = true)
 |-- peer_interaction: boolean (nullable = true)
 |-- age: integer (nullable = true)
 |-- education_level: string (nullable = true)
 |-- study_field: string (nullable = true)
 |-- learning_style: string (nullable = true)
 |-- device_preference: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- previous_online_courses: integer (nullable = true)
 |-- has_learning_disability: integer (nullable = true)
 |-- motivation_level: string (nullable = true)
 |-- course_title: string (nullable = true)
 |

In [8]:
df_spark.show(5)

+------------------+-------------+----------+-------------------+-----------------+----------------+---------------+-----+--------+-----------+-----------+----------------+---+---------------+-----------+---------------+-----------------+--------+-----------------------+-----------------------+----------------+--------------------+----------------+----------------+--------------+-----------------+-------------+---------------+-----------+--------------------+-------------------+----------------+-----------------+----------------+----------------+---------------------+
|    interaction_id|   student_id| course_id|          timestamp|    activity_type|duration_seconds|completion_rate|score|attempts|device_used|help_sought|peer_interaction|age|education_level|study_field| learning_style|device_preference|timezone|previous_online_courses|has_learning_disability|motivation_level|        course_title|    subject_area|difficulty_level|duration_weeks|instructor_rating|course_format|has_assignm

In [9]:
df_spark.describe().show()

+-------+------------------+-------------+----------+-----------------+-----------------+-------------------+-------------------+------------------+-----------+------------------+---------------+---------------+--------------+-----------------+--------+-----------------------+-----------------------+----------------+--------------+------------+----------------+-----------------+-------------------+-------------+------------------+------------------+--------------------+-------------------+------------------+------------------+-----------------+-----------------+---------------------+
|summary|    interaction_id|   student_id| course_id|    activity_type| duration_seconds|    completion_rate|              score|          attempts|device_used|               age|education_level|    study_field|learning_style|device_preference|timezone|previous_online_courses|has_learning_disability|motivation_level|  course_title|subject_area|difficulty_level|   duration_weeks|  instructor_rating|course_f

In [10]:
print("Número de filas:", df_spark.count())
print("Número de columnas:", len(df_spark.columns))

Número de filas: 200000
Número de columnas: 36


## 6. Transformaciones con PySpark

En esta sección se aplican tres transformaciones diferentes sobre el DataFrame de Spark:

- `filter`
- `groupBy`
- `orderBy`

El objetivo es adaptar el procesamiento al contexto de EdTech Learning Analytics.

### 6.1 Filtrado de interacciones con rendimiento menor al promedio

Se identifican actividades evaluables donde el `score` obtenido es menor que el promedio general del estudiante.

In [11]:
df_bajo_promedio = (
    df_spark
    .filter(
        col("score").isNotNull()
        & (col("score") < col("average_student_score"))
    )
)

print(
    "Interacciones evaluables por debajo del promedio del estudiante:",
    df_bajo_promedio.count()
)

df_bajo_promedio.select(
    "student_id",
    "course_title",
    "activity_type",
    "score",
    "average_student_score"
).show(10)

Interacciones evaluables por debajo del promedio del estudiante: 33178
+-------------+---------------+-----------------+-----+---------------------+
|   student_id|   course_title|    activity_type|score|average_student_score|
+-------------+---------------+-----------------+-----+---------------------+
|student_00034|    Biology 541|     Quiz Attempt| 0.66|               0.7612|
|student_05133|  Economics 371|Assignment Submit| 0.68|                 0.77|
|student_03171| Psychology 217|     Quiz Attempt| 0.68|                0.745|
|student_06432|Mathematics 376|Assignment Submit| 0.68|                0.784|
|student_04178|      Music 110|     Quiz Attempt| 0.74|               0.7675|
|student_04821|Mathematics 268|     Quiz Attempt|  0.7|               0.8008|
|student_01814|    Physics 282|     Quiz Attempt| 0.63|               0.7783|
|student_05195|    History 244|     Quiz Attempt| 0.62|               0.7133|
|student_06539|  Economics 371|Assignment Submit| 0.79|               0

### 6.2 Rendimiento académico por curso

Se agrupan las interacciones por curso para calcular el promedio de `score`, el promedio de `completion_rate` y la cantidad de interacciones registradas.

In [12]:
rendimiento_por_curso = (
    df_spark
    .groupBy(
        "course_id",
        "course_title"
    )
    .agg(
        avg("score").alias("average_score"),
        avg("completion_rate").alias("average_completion_rate"),
        count("*").alias("total_interactions")
    )
)

rendimiento_por_curso.show(10)

+----------+--------------------+------------------+-----------------------+------------------+
| course_id|        course_title|     average_score|average_completion_rate|total_interactions|
+----------+--------------------+------------------+-----------------------+------------------+
|course_011|      Psychology 251|0.7954586129753913|     0.8134092634776008|              1317|
|course_090|         History 318|0.7961009174311928|     0.8053303637713437|              1347|
|course_123|         History 489|0.7961505376344086|     0.8104820143884894|              1390|
|course_045|Language Learning...|0.7984999999999999|     0.7968332108743571|              1361|
|course_120|             Art 472|0.7949230769230771|     0.8320390455531453|              1383|
|course_107|         Physics 282|0.7959753086419753|     0.8136396790663749|              1371|
|course_071|     Mathematics 489|0.7968211920529801|     0.8073544583640383|              1357|
|course_093|           Music 110|0.79832

### 6.3 Ordenamiento de cursos por rendimiento

Se ordenan los cursos según su promedio académico para identificar aquellos con mayor rendimiento.

In [13]:
cursos_ordenados = (
    rendimiento_por_curso
    .orderBy(
        col("average_score").desc()
    )
)

cursos_ordenados.show(10)

+----------+--------------------+------------------+-----------------------+------------------+
| course_id|        course_title|     average_score|average_completion_rate|total_interactions|
+----------+--------------------+------------------+-----------------------+------------------+
|course_079|         Physics 202|  0.81636563876652|     0.8095620437956204|              1370|
|course_149|Language Learning...|0.8140047393364929|     0.7948678943154525|              1249|
|course_109|         Physics 165|0.8120813397129185|     0.8164371497197758|              1249|
|course_083|       Chemistry 598|0.8114939759036144|     0.8058661710037174|              1345|
|course_003|Computer Science 549|0.8112826086956522|     0.8133854166666667|              1344|
|course_103|      Psychology 508|0.8104941176470588|     0.7915408085430969|              1311|
|course_034|       Economics 125|0.8100945626477541|     0.8065950920245398|              1304|
|course_069|       Economics 208|0.80941

### Resultado de las transformaciones

Se aplicaron correctamente las tres transformaciones solicitadas con PySpark:

- `filter` para identificar interacciones con rendimiento inferior al promedio del estudiante,
- `groupBy` para calcular indicadores por curso,
- `orderBy` para ordenar los cursos según su rendimiento promedio.

## 7. Vista temporal y consulta con Spark SQL

En esta sección se crea una vista temporal a partir del DataFrame de Spark.

Posteriormente se ejecuta una consulta SQL para analizar el rendimiento académico por curso y nivel de dificultad.

In [14]:
df_spark.createOrReplaceTempView("edtech_interactions")

print("Vista temporal creada correctamente")

Vista temporal creada correctamente


### 7.1 Consulta SQL

Se utiliza Spark SQL para calcular indicadores académicos agrupados por curso y nivel de dificultad.

In [15]:
resultado_sql = spark.sql("""
SELECT
    course_id,
    course_title,
    difficulty_level,
    COUNT(*) AS total_interactions,
    AVG(score) AS average_score,
    AVG(completion_rate) AS average_completion_rate
FROM edtech_interactions
GROUP BY
    course_id,
    course_title,
    difficulty_level
ORDER BY
    average_score DESC
""")

resultado_sql.show(10)

+----------+--------------------+----------------+------------------+------------------+-----------------------+
| course_id|        course_title|difficulty_level|total_interactions|     average_score|average_completion_rate|
+----------+--------------------+----------------+------------------+------------------+-----------------------+
|course_079|         Physics 202|        Beginner|              1370|  0.81636563876652|     0.8095620437956204|
|course_149|Language Learning...|    Intermediate|              1249|0.8140047393364929|     0.7948678943154525|
|course_109|         Physics 165|        Beginner|              1249|0.8120813397129185|     0.8164371497197758|
|course_083|       Chemistry 598|    Intermediate|              1345|0.8114939759036144|     0.8058661710037174|
|course_003|Computer Science 549|    Intermediate|              1344|0.8112826086956522|     0.8133854166666667|
|course_103|      Psychology 508|        Advanced|              1311|0.8104941176470588|     0.7

### Resultado de la consulta SQL

Se creó una vista temporal en Spark y se ejecutó una consulta SQL para obtener el promedio académico, nivel de finalización y número de interacciones por curso.

La consulta permite demostrar el uso de Spark SQL sobre el mismo dataset procesado mediante PySpark.

## 8. Comparación de rendimiento entre Pandas y Spark

En esta sección se compara el tiempo de ejecución de una misma operación utilizando Pandas y PySpark.

La operación seleccionada consiste en calcular el promedio de `score` por curso.

In [16]:
df_pandas = pd.read_csv(ruta_dataset_limpio)

inicio_pandas = time.perf_counter()

resultado_pandas = (
    df_pandas
    .groupby("course_title")["score"]
    .mean()
)

fin_pandas = time.perf_counter()

tiempo_pandas = fin_pandas - inicio_pandas

print(
    f"Tiempo de ejecución con Pandas: "
    f"{tiempo_pandas:.6f} segundos"
)

Tiempo de ejecución con Pandas: 0.011951 segundos


In [17]:
inicio_spark = time.perf_counter()

resultado_spark = (
    df_spark
    .groupBy("course_title")
    .agg(
        avg("score").alias("average_score")
    )
    .collect()
)

fin_spark = time.perf_counter()

tiempo_spark = fin_spark - inicio_spark

print(
    f"Tiempo de ejecución con Spark: "
    f"{tiempo_spark:.6f} segundos"
)

Tiempo de ejecución con Spark: 0.263100 segundos


In [18]:
comparacion_tiempos = pd.DataFrame({
    "tecnologia": [
        "Pandas",
        "Spark"
    ],
    "tiempo_segundos": [
        tiempo_pandas,
        tiempo_spark
    ]
})

display(comparacion_tiempos)

,tecnologia,tiempo_segundos
0,Pandas,0.011951
1,Spark,0.263100


### Interpretación de la comparación

La comparación permite observar el tiempo requerido por Pandas y Spark para ejecutar una operación equivalente.

Debido a que el dataset utilizado puede procesarse completamente en la memoria de una sola computadora, es posible que Pandas presente un tiempo menor.

Spark incorpora costos adicionales asociados con la planificación y ejecución distribuida de tareas, por lo que sus ventajas se vuelven más evidentes cuando se trabaja con volúmenes de datos mucho mayores o con múltiples nodos.

## 9. Almacenamiento en formato Parquet

En esta sección se guarda el DataFrame procesado por Spark en formato Parquet.

Parquet es un formato columnar eficiente para almacenamiento y procesamiento de grandes volúmenes de datos.

In [19]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EdTechLearningAnalytics_BigData")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config(
        "spark.hadoop.fs.defaultFS",
        "file:///"
    )
    .getOrCreate()
)

print("Spark session creada correctamente")
print("Versión de Spark:", spark.version)

Spark session creada correctamente
Versión de Spark: 4.3.0


In [20]:
import sys
import os

print("Python:", sys.executable)
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

Python: c:\venvs\edtech-spark\Scripts\python.exe
HADOOP_HOME: None


In [21]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EdTechLearningAnalytics_BigData")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.hadoop.fs.defaultFS", "file:///")
    .config("spark.local.dir", r"C:\spark-temp")
    .getOrCreate()
)

print("Spark:", spark.version)

Spark: 4.3.0


In [22]:
import os

print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print("Existe C:\\hadoop:", os.path.exists(r"C:\hadoop"))
print("Existe bin:", os.path.exists(r"C:\hadoop\bin"))
print(
    "Existe winutils:",
    os.path.exists(r"C:\hadoop\bin\winutils.exe")
)

HADOOP_HOME: None
Existe C:\hadoop: True
Existe bin: True
Existe winutils: False


In [23]:
df_spark.write \
    .mode("overwrite") \
    .parquet(str(ruta_parquet))

Py4JJavaError: An error occurred while calling o110.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir

In [ ]:
try:
    (
        df_spark.write
        .mode("overwrite")
        .parquet(str(ruta_parquet))
    )

except Exception as e:
    print("Tipo:", type(e).__name__)
    print("\nMensaje:")
    print(str(e))

Tipo: Py4JJavaError

Mensaje:
An error occurred while calling o116.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: Could not locate Hadoop executable: C:\hadoop\bin\winutils.exe -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(Raw

In [ ]:
from pathlib import Path
import pyspark

pyspark_dir = Path(pyspark.__file__).parent
jars_dir = pyspark_dir / "jars"

hadoop_jars = sorted(
    p.name
    for p in jars_dir.glob("hadoop-*.jar")
)

for jar in hadoop_jars:
    print(jar)

hadoop-client-api-3.5.0.jar
hadoop-client-runtime-3.5.0.jar


### Exportación a formato Parquet

Se intentó guardar el DataFrame procesado con PySpark en formato Parquet mediante:

```python
df_spark.write \
    .mode("overwrite") \
    .parquet(str(ruta_parquet))
```

Sin embargo, la operación no pudo completarse debido a una limitación del entorno local en Windows. Durante la escritura, Spark utiliza componentes de Hadoop para administrar el sistema de archivos y el proceso requiere la utilidad `winutils.exe`, la cual no se encuentra disponible en la configuración actual.

El DataFrame fue cargado, explorado y transformado correctamente con PySpark; por lo tanto, el inconveniente corresponde únicamente a la persistencia del resultado en formato Parquet y no a las transformaciones realizadas ni a la calidad de los datos.

Para efectos de esta práctica, se deja documentada esta limitación técnica y se continúa con las siguientes actividades.